In [2]:
!pip install arxiv

In [3]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

/Users/jaideeptrip/Desktop/LANGCHAIN/LangChain/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
!pip install  wikipedia
api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

  Using cached wikipedia-1.4.0-py3-none-any.whl


In [5]:
wiki.name

'wikipedia'

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

In [7]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OpenAIEmbeddings

loader = WebBaseLoader('https://docs.langchain.com/langsmith/home')
docs = loader.load()
documents= RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents,OpenAIEmbeddings())
retriever = vectordb.as_retriever()
retriever


USER_AGENT environment variable not set, consider setting it to identify your requests.
/var/folders/1_/03b51ty51gs0gkzh0w_t217w0000gn/T/ipykernel_9198/3026122718.py:10: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  vectordb = FAISS.from_documents(documents,OpenAIEmbeddings())


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x110f71130>, search_kwargs={})

In [8]:
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",   # underscore, no spaces, correct spelling
    "Search for information about Langsmith. For any questions about Langsmith, you must use this tool"
)

NameError: name 'create_retriever_tool' is not defined

In [ ]:
retriever_tool.name

In [ ]:
##arxiv
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

In [ ]:
tools = [wiki,arxiv,retriever_tool]

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model ="gpt-3.5-turbo-0125",temperature=0)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [ ]:
import langchain
import langchain_core
import langchain_community
print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print("langchain_community:", langchain_community.__version__)

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor


agent = create_tool_calling_agent(llm, tools, prompt)



In [ ]:
from langchain.agents import AgentExecutor
agent_executor= AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

In [ ]:
agent_executor.invoke({"input":"Tell me about Langsmith"})

In [ ]:
agent_executor.invoke({"input":"Tell me about machine learning"})